# Lab 8.5 &mdash; Parallel Fan-Out

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Run every specialist on the SAME customer ticket at once
- Collect each result tagged with the agent that produced it
- Survive one agent failing -- fault tolerance in a fan-out

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; routing, synthesis, tool wiring, agent structure &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Parallel — fan-out for coverage & speed](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-08-05"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
In the **parallel** (fan-out) shape, several agents work the **same input** at once and their outputs
are combined (deck slide 10). For a support ticket you fan it out to the **billing, tech and policy**
specialists together &mdash; each an independent lens, so between them they catch what one alone would
miss. Two practical rules: keep each result **tagged with the agent** that produced it (you must know
who said what), and make the fan-out **fault-tolerant** &mdash; if one specialist is down, the others
still return. But fan-out creates a new problem &mdash; you now have **several outputs and need one**
&mdash; which is the decision-making you'll build next.

In [ ]:
# Three specialists, each a different lens on the SAME ticket -- and one of them is currently DOWN.
def billing_agent(t):
    return "duplicate charge on 4471" if "charg" in t.lower() else "no billing issue"
def tech_agent(t):
    return "matches BUG-231" if "crash" in t.lower() else "no tech issue"
def policy_agent(t):
    raise RuntimeError("policy service unavailable")   # this specialist is DOWN -> the fan-out must survive it
SPECIALISTS = {"billing": billing_agent, "tech": tech_agent, "policy": policy_agent}
print("fan-out targets:", list(SPECIALISTS))

## Your Turn
Complete `fan_out`: run every specialist on the **same** ticket, tag each result by agent, and keep
going when one raises.

In [ ]:
def fan_out(ticket, specialists):
    results = {}
    for name, agent in specialists.items():
        try:
            results[name] = ___   # TODO: run THIS agent on the ticket (same input for all)
        except Exception as e:
            results[name] = ___   # TODO: this agent is down -- record a marker string beginning "ERROR: " (include type(e).__name__) so the fan-out survives
    return results

try:
    out = fan_out('charged twice and the app keeps crashing', SPECIALISTS)
    for name, res in out.items():
        print(f'{name:8}: {res}')
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("fan-out returns one result per specialist", lambda: len(fan_out("charged twice and crash", SPECIALISTS)) == 3)
expect_true("each result is tagged by agent identity", lambda: set(fan_out("x", SPECIALISTS)) == {"billing", "tech", "policy"})
expect_true("every specialist saw the same ticket", lambda: fan_out("charged twice and crash", SPECIALISTS)["billing"] == "duplicate charge on 4471" and fan_out("charged twice and crash", SPECIALISTS)["tech"] == "matches BUG-231")
expect_true("a failing specialist does NOT crash the fan-out", lambda: fan_out("x", SPECIALISTS)["policy"].startswith("ERROR"))
expect_true("the surviving specialists still returned findings", lambda: not fan_out("charged twice and crash", SPECIALISTS)["billing"].startswith("ERROR"))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Fan-out buys coverage and speed -- latency is the slowest agent, not the sum -- and staying tagged + fault-tolerant means one agent going down doesn't take the team with it. But now you have several outputs and need one: that convergence is decision making, coming up.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>